# 03 — Earnings previews: the ex ante information set

Notebook 02 covered the ten **facts** distilled from the earnings call — what the
company disclosed. This notebook covers the other item an event's `disclosure` can
carry: the **earnings preview**, an agent-written research note assembled from public
sources *before* the release. Think of it as the information forming the market's ex
ante expectation about the announcement: consensus, guidance, what moved the stock
last time, and what to watch.

It sits in `disclosure.items[]` as `id="earnings-preview"`, `kind="text"`,
`source="claude_code_web_research"`, `media_type="text/markdown"`, with `content` a
single markdown string. It is **optional** — events from before it was disseminated,
and events for which no note was produced, have none.

This notebook:

1. What you receive — coverage in the 2026Q3 archive, and six events to explore
2. How a preview is built (at a high level)
3. When it is built — and the caveat for the 2026Q3 archive
4. Anatomy — the note's skeleton across the six
5. One note in full
6. The other five
7. Ex ante vs. ex post — preview, facts, surprise, and the realized reaction

With a live key it reads the real 2026Q3 archive; without one it uses six **real**
archive lines bundled in `data/sample/` (see `data/README.md`), so everything below
runs either way.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from examples import Client, load_config
from examples.archive import download_archive, read_jsonl_gz
from examples.preview import (
    PREVIEW_PLUGIN,
    PREVIEW_PLUGIN_REPO,
    PREVIEW_SKILL,
    PREVIEW_SKILL_PATH,
    earnings_preview_from_disclosure,
    preview_display_markdown,
    preview_sections,
    preview_title,
    previews_frame,
)
from examples.scoring import outcomes_frame
from examples.summary import facts_from_disclosure

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache
SAMPLE_DIR = REPO / "data" / "sample"

QUARTER = "2026Q3"  # the first quarter with previews
TICKERS = ["NVDA", "WMT", "HD", "CRM", "AVGO", "ORCL"]

config = load_config()
print("Mode:", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")

## 1. What you receive

Previews start partway through 2026Q3 — the archive quarter in which dissemination
began — so earlier events, and the first weeks of that quarter, carry none. Load the
one file and count.

In live mode this pulls just the 2026Q3 file. The quarter is still unsealed, so the
file grows as events are scored; `download_archive` compares sizes against the
manifest and re-downloads when the cache is stale.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        manifest = client.archive_manifest()
        (path,) = download_archive(
            manifest,
            ARCHIVE_DIR,
            client=client,
            only=lambda f: f.event_type == "EARNINGS_RELEASE" and f.quarter == QUARTER,
        )
else:
    path = SAMPLE_DIR / f"archive_EARNINGS_RELEASE_{QUARTER}.jsonl.gz"
    print(f"Sample mode: reading {path.name}")

records = list(read_jsonl_gz(path))
coverage = previews_frame(records)

n_with = int(coverage["has_preview"].sum())
print(f"{len(records):,} events in {QUARTER}; {n_with:,} carry a preview")
if n_with:
    first = coverage.loc[coverage["has_preview"], "event_datetime"].min()
    print(f"Earliest event with a preview: {first:%Y-%m-%d}")

Now the six events we will read. Each is the ticker's one 2026Q3 event (the latest,
should a ticker ever have more than one in a quarter).

In [ ]:
def latest_record(ticker: str) -> dict | None:
    hits = [r for r in records if r["focal_assets"][0]["identifier_value"] == ticker]
    return max(hits, key=lambda r: r["event_datetime"]) if hits else None


chosen = {t: latest_record(t) for t in TICKERS}
missing = [t for t, r in chosen.items() if r is None]
if missing:
    print(f"Not in this file: {missing}")
chosen = {t: r for t, r in chosen.items() if r is not None}

overview = previews_frame(chosen.values()).set_index("tickers").loc[list(chosen)]
display(
    overview[["event_id", "event_datetime", "knowledge_cutoff", "has_preview", "preview_chars"]]
)

## 2. How a preview is built

The competition does not write the preview itself. It runs Anthropic's open
**`earnings-reviewer` plugin** — specifically its **`earnings-preview` skill** — through
Claude Code, pinned to one commit of the public repository below so that every note in
a period comes from the same skill text.

The agent receives a one-line request naming the company, its ticker, the fiscal
period and the scheduled event time, and has **web search and web fetch as its only
tools**. Nothing else is injected: no transcript, no consensus feed, no market data, no
competition data. Whatever the note knows, the agent went and found — and the skill
instructs it to cite the source and date of each estimate, to say when something
(an options-implied move, a whisper number) could not be found, and never to
estimate in its place.

The skill prescribes the note's shape, which is why the six read alike:

- the company, quarter and event date up top
- a **consensus estimates** table (and the company's own guidance)
- the **key metrics to watch**, ranked
- a **bull / base / bear** scenario table
- a **catalyst checklist** for the call
- the **trading setup**: recent performance, historical earnings reactions, the
  options-implied move, positioning

In production the skill runs on Claude Opus 5 at high effort. Scheduling and
infrastructure are out of scope here — the point is that the note is a public-sources
research product with a fixed brief, not a proprietary signal.

In [ ]:
print(f"plugin : {PREVIEW_PLUGIN}")
print(f"skill  : {PREVIEW_SKILL}")
print(f"source : {PREVIEW_PLUGIN_REPO}")
print(f"         {PREVIEW_SKILL_PATH}")

## 3. When it is built

Every event has a `knowledge_cutoff` (notebook 00): the instant after which an agent
must not use information, and the moment the one-day abnormal-return window (`car1`)
opens — the prior close for a pre-market reporter, the same-day close for an
after-close reporter.

**Going forward, a disseminated preview is assembled before that cutoff**, so it is a
clean ex ante object: nothing in it post-dates the start of the measurement window.

**The 2026Q3 archive is the exception.** Those previews were produced live, ahead of
each release, and attached to the archive afterwards — but the run was not yet keyed
to the cutoff, so a note for a pre-market reporter may have been finished the evening
before or the morning of, after the prior close. No timestamps travel with the item,
so you cannot tell which from the archive alone — though a note sometimes says so
itself (WMT's below opens with a timing caveat that the release is minutes away).
Treat the 2026Q3 previews as illustrative of what you will receive, not as strictly
cutoff-clean inputs for a backtest.

## 4. Anatomy

`preview_title` and `preview_sections` pull the `#` title and the `##` headings, which
is enough to see the skill's skeleton — and how far each note departs from it.

In [ ]:
previews = {t: earnings_preview_from_disclosure(r) for t, r in chosen.items()}
previews = {t: p for t, p in previews.items() if p is not None}

rows = [
    {"ticker": t, "title": preview_title(p), "n": i, "section": s}
    for t, p in previews.items()
    for i, s in enumerate(preview_sections(p), start=1)
]
pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame(rows).set_index(["ticker", "title", "n"]))

The common spine — consensus, key metrics, scenarios, catalysts, trading setup — is
the skill's brief. The extras are the agent's judgment about what *this* event turns
on: a "setup" paragraph, a section of guidance arithmetic, a data-quality note.

## 5. One note in full

NVIDIA's Q2 FY2027 preview. (Dollar signs are escaped for display only — the
disseminated string is plain markdown.)

In [ ]:
def show(ticker: str) -> None:
    record, preview = chosen[ticker], previews[ticker]
    display(
        Markdown(
            f"---\n**{ticker}** · `{record['event_id']}` · event {record['event_datetime']} · "
            f"knowledge cutoff {record['knowledge_cutoff']} · {len(preview):,} chars\n\n"
            + preview_display_markdown(preview)
        )
    )


show("NVDA")

## 6. The other five

Read them for the range: a chip name where the note argues the next-quarter guide is
the whole event (AVGO), two retailers whose prints are a read on the consumer (WMT,
HD), and two software names where the note says the P&L is not the point — CRM's
guidance arithmetic, ORCL's backlog and capex.

In [ ]:
for ticker in previews:
    if ticker != "NVDA":
        show(ticker)

## 7. Ex ante vs. ex post

Each archive line lets you put the three pieces side by side:

- the **preview** — what a careful reader could know going in,
- the **facts** — what the company then disclosed on the call, and
- the **reaction** — the earnings surprise and the realized one-day abnormal return
  (`car1`), which is what the competition scores.

`examples.scoring.outcomes_frame` pulls the surprise and `car1` per (event, asset);
we deliberately do **not** call `add_percentiles` here — the competition ranks across
a whole quarter, and percentiles over six events would be meaningless.

In [ ]:
outcomes = outcomes_frame(chosen.values())  # one row per (event, asset) that has a car1
outcomes = (
    outcomes.set_index("identifier_value")[["surprise", "car1"]]
    if not outcomes.empty
    else pd.DataFrame(columns=["surprise", "car1"])
)

table = pd.DataFrame(
    {
        "event_datetime": {t: r["event_datetime"] for t, r in chosen.items()},
        "knowledge_cutoff": {t: r["knowledge_cutoff"] for t, r in chosen.items()},
        "preview_chars": {t: len(previews.get(t) or "") for t in chosen},
        "n_facts": {t: len(facts_from_disclosure(r)) for t, r in chosen.items()},
    }
).join(outcomes, how="left")  # a very recent live event may have no car1 yet

display(table.style.format({"surprise": "{:+.4f}", "car1": "{:+.2%}"}, na_rep="\u2014"))

Pick one and read the three together — for CRM, say, the preview's "what would it
take" arithmetic, the facts that answered it, and a double-digit `car1` — and you have
the competition's problem in miniature: given the expectation and the disclosure,
where in the quarter's cross-section will the reaction land?

## What to take away

- The preview is **optional** and arrived in 2026Q3. Match on `id` / `kind`, never on
  position, and build for its absence.
- It is a **public-sources research note with a fixed brief** — Anthropic's open
  `earnings-preview` skill, run with web search as the only tool — not a model
  prediction and not a proprietary feed. Its sources and dates are cited inline.
- Going forward it is assembled **before the CAR1 window opens**; the 2026Q3 archive
  previews are not guaranteed to be, so treat them as illustrative.
- The facts (notebook 02) are the ex post disclosure; the preview is the ex ante
  expectation; `car1` is the reaction. `examples.scoring` turns the last into the
  percentile target the competition scores.